### Time Series Pattern Classification

#### Objective
The goal of this assignment is to analyze time-series data of shape `(batch, 10, 2)`, discover repeating temporal patterns in **class 1**, and build a binary classification model that can distinguish class `0` and class `1` samples.

#### Assignment Requirements
The notebook covers:
- Feature Analysis
- Model Training
- Model Selection
- Model Evaluation
- Production Readiness and Limitations

#### Important Constraint
SMOTE is not used because the problem contains temporal dependencies, and synthetic oversampling could distort the sequence structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

Load Data

In [5]:
X = np.load("../data/X.npz")["arr_0"]
y = np.load("../data/y.npz")["arr_0"]

print("Shape of X: ", X.shape)
print("Shape of y: ", y.shape)

Shape of X:  (999991, 10, 2)
Shape of y:  (999991,)


Basic Inspection

In [6]:
print("First sample shape:", X[0].shape)
print("\nFirst sample:\n", X[0])
print("\nUnique labels:", np.unique(y))
print("\nClass counts:", Counter(y))

First sample shape: (10, 2)

First sample:
 [[  7.37123228  -1.94677024]
 [-18.42842376  -5.09767874]
 [ -1.43933599  -9.91950072]
 [ -2.79447095  -9.62893966]
 [-18.08327104  16.70333388]
 [-19.03699992  17.74685937]
 [ -3.16413771  -1.02089318]
 [  6.33031237  -2.01437082]
 [-10.59985909  -2.1739294 ]
 [ -3.5635373   -8.66233473]]

Unique labels: [0. 1.]

Class counts: Counter({np.float64(0.0): 999298, np.float64(1.0): 693})


Class distribution table

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()                             #how many samples belong to each class
class_percent = pd.Series(y).value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percent.round(4)
})

class_summary.index.name = "class"
class_summary

,count,percentage
class,,
0.0,999298,99.9307
1.0,693,0.0693


#### Initial Observations

- The dataset contains a very large number of samples.
- Each sample is a short multivariate time series with 10 time steps and 2 dimensions.
- The target is binary: class 0 and class 1.
- Class 1 is extremely rare compared to class 0, so the dataset is highly imbalanced.
- Because the minority class is rare and the data is temporal, the analysis must focus on discovering stable sequence patterns rather than relying on synthetic oversampling.

Separate class 0 and class 1 indices

In [9]:
#position of class samples in dataset
class0_idx = np.where(y == 0)[0]        #np.where() returns a tuple, (array([0,2,3]),)  |  So [0] extracts the array inside, [0,2,3]
class1_idx = np.where(y == 1)[0]

print("Number of class 0 samples:", len(class0_idx))
print("Number of class 1 samples:", len(class1_idx))

Number of class 0 samples: 999298
Number of class 1 samples: 693


Inspect a few class 1 samples

In [10]:
for i in range(3):
    print(f"\nClass 1 sample index: {class1_idx[i]}")
    print(X[class1_idx[i]])


Class 1 sample index: 7882
[[ -2.75303681  -4.6491321 ]
 [ -3.45476136  -9.03654515]
 [ 11.87789865   0.55809885]
 [-15.75800829  -6.80038371]
 [ 14.0137616  -12.16455052]
 [ 20.05801103  -5.1241688 ]
 [ -4.05498555 -10.79821021]
 [ -4.23520959  -1.31881298]
 [ -3.831798    -1.76211803]
 [ -4.21602768 -11.44281674]]

Class 1 sample index: 7883
[[ -3.45476136  -9.03654515]
 [ 11.87789865   0.55809885]
 [-15.75800829  -6.80038371]
 [ 14.0137616  -12.16455052]
 [ 20.05801103  -5.1241688 ]
 [ -4.05498555 -10.79821021]
 [ -4.23520959  -1.31881298]
 [ -3.831798    -1.76211803]
 [ -4.21602768 -11.44281674]
 [ -8.86038996  -3.09925031]]

Class 1 sample index: 7884
[[ 11.87789865   0.55809885]
 [-15.75800829  -6.80038371]
 [ 14.0137616  -12.16455052]
 [ 20.05801103  -5.1241688 ]
 [ -4.05498555 -10.79821021]
 [ -4.23520959  -1.31881298]
 [ -3.831798    -1.76211803]
 [ -4.21602768 -11.44281674]
 [ -8.86038996  -3.09925031]
 [-16.99173139  -5.78773324]]
